In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%H:%M:%S")

from src.ml.features import (
    NUMERIC_FEATURES, LABEL_COLUMNS, ID_COLUMNS, get_feature_count
)

print("Feature group breakdown:")
for group, count in get_feature_count().items():
    print(f"  {group:<15} {count:>3}")

print(f"\nTotal feature columns: {len(NUMERIC_FEATURES)}")
print(f"Label columns:         {LABEL_COLUMNS}")
print(f"ID columns:            {ID_COLUMNS}")

Feature group breakdown:
  volumetric       10
  packet_length    16
  iat              14
  flags            12
  headers           2
  bulk              6
  subflow           4
  window            4
  timing            8
  categorical       3
  total_numeric    76

Total feature columns: 76
Label columns:         ['is_attack', 'attack_family_denorm']
ID columns:            ['event_id', 'event_time', 'date_sk', 'time_sk', 'src_asset_sk', 'dest_asset_sk']


In [2]:
from src.ml.data import load_labeled_data

# Use 25K rows per family for now — manageable size
df = load_labeled_data(sample_per_family=25_000)
print(f"\nLoaded shape: {df.shape}")
print(f"\nClass distribution:")
print(df["attack_family_denorm"].value_counts())
print(f"\nBinary distribution:")
print(df["is_attack"].value_counts())

16:53:55 | Loading labeled events (≤25000/family)
16:54:33 | Loaded 118,028 labeled rows
16:54:33 | Class distribution: {'Benign': 25000, 'DoS': 25000, 'DDoS': 25000, 'Reconnaissance': 25000, 'Brute Force': 13835, 'Web Attack': 2180, 'Botnet': 1966, 'Infiltration': 36, 'Exploit': 11}



Loaded shape: (118028, 84)

Class distribution:
attack_family_denorm
Benign            25000
DoS               25000
DDoS              25000
Reconnaissance    25000
Brute Force       13835
Web Attack         2180
Botnet             1966
Infiltration         36
Exploit              11
Name: count, dtype: int64

Binary distribution:
is_attack
1    93028
0    25000
Name: count, dtype: int64


In [4]:
import pandas as pd

print("=== Missing values per feature ===")
missing = df[NUMERIC_FEATURES].isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
worst = pd.DataFrame({"missing": missing, "pct": missing_pct}).sort_values("pct", ascending=False)
print(worst.head(10))

print("\n=== Infinity values ===")
import numpy as np
inf_cols = (df[NUMERIC_FEATURES] == np.inf).sum() + (df[NUMERIC_FEATURES] == -np.inf).sum()
print(inf_cols[inf_cols > 0].sort_values(ascending=False))

print("\n=== Numeric ranges (first 10 features) ===")
print(df[NUMERIC_FEATURES[:10]].describe().round(2))

=== Missing values per feature ===
                      missing   pct
flow_packets_per_sec      139  0.12
flow_bytes_per_sec        139  0.12
flow_duration               0  0.00
total_fwd_packets           0  0.00
total_length_fwd            0  0.00
total_bwd_packets           0  0.00
total_length_bwd            0  0.00
fwd_pkts_per_sec            0  0.00
bwd_pkts_per_sec            0  0.00
down_up_ratio               0  0.00

=== Infinity values ===
Series([], dtype: int64)

=== Numeric ranges (first 10 features) ===
       flow_duration  total_fwd_packets  total_bwd_packets  total_length_fwd  \
count   1.180280e+05          118028.00          118028.00         118028.00   
mean    1.846557e+07               5.71               5.57            441.26   
std     3.533724e+07              80.97             106.49          14773.80   
min     0.000000e+00               1.00               0.00              0.00   
25%     6.000000e+01               1.00               1.00              2.0

In [5]:
from src.ml.data import get_train_val_test_splits

splits = get_train_val_test_splits(sample_per_family=25_000)

print("Split shapes:")
for name in ("train", "val", "test"):
    X = splits[f"X_{name}"]
    y = splits[f"y_{name}"]
    print(f"  {name:<6} X: {X.shape}, y: {y.shape}")

print("\nClass distribution in training set:")
print(splits["y_train"]["attack_family_denorm"].value_counts())

print("\nBinary distribution in training set:")
print(splits["y_train"]["is_attack"].value_counts(normalize=True).round(3))

16:55:47 | Loading labeled events (≤25000/family)
16:56:23 | Loaded 118,028 labeled rows
16:56:23 | Class distribution: {'Benign': 25000, 'DoS': 25000, 'DDoS': 25000, 'Reconnaissance': 25000, 'Brute Force': 13835, 'Web Attack': 2180, 'Botnet': 1966, 'Infiltration': 36, 'Exploit': 11}
16:56:24 | Splits: train=82,619, val=11,803, test=23,606


Split shapes:
  train  X: (82619, 76), y: (82619, 8)
  val    X: (11803, 76), y: (11803, 8)
  test   X: (23606, 76), y: (23606, 8)

Class distribution in training set:
attack_family_denorm
DDoS              17500
Reconnaissance    17500
DoS               17500
Benign            17500
Brute Force        9684
Web Attack         1526
Botnet             1376
Infiltration         25
Exploit               8
Name: count, dtype: int64

Binary distribution in training set:
is_attack
1    0.788
0    0.212
Name: proportion, dtype: float64


In [6]:
# Confirm no overlap between splits by event_id
train_ids = set(splits["y_train"]["event_id"])
val_ids   = set(splits["y_val"]["event_id"])
test_ids  = set(splits["y_test"]["event_id"])

print(f"Train ∩ Val:  {len(train_ids & val_ids)}  (expected 0)")
print(f"Train ∩ Test: {len(train_ids & test_ids)}  (expected 0)")
print(f"Val ∩ Test:   {len(val_ids & test_ids)}  (expected 0)")
print(f"\n✅ No data leak across splits" if not (train_ids & val_ids or train_ids & test_ids or val_ids & test_ids) else "❌ DATA LEAK DETECTED")

Train ∩ Val:  0  (expected 0)
Train ∩ Test: 0  (expected 0)
Val ∩ Test:   0  (expected 0)

✅ No data leak across splits
